# SQL Model Setup

This notebook is used to create and test the SQL schema for campaigns and shopify sales data.

In [28]:
import sqlite3
import pandas as pd

# connect to database (use ONE combined DB now)
conn = sqlite3.connect("../sql/final_marketing.db")

## Load cleaned datasets into SQL

In [29]:
# load campaigns DB

conn1 = sqlite3.connect("../sql/cleaned_campaigns.db")
df_campaign = pd.read_sql("SELECT * FROM campaign_cleaned", conn1)

# save into final DB
df_campaign.to_sql("campaigns", conn, if_exists="replace", index=False)

10028

In [30]:
# load shopify DB
conn2 = sqlite3.connect("../sql/cleaned_shopify.db")
df_shopify = pd.read_sql("SELECT * FROM shopify_sales", conn2)

# save into final DB
df_shopify.to_sql("shopify", conn, if_exists="replace", index=False)

5659

In [31]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

,name
0,dim_date
1,dim_campaign
2,campaigns
3,shopify


In [32]:
query = "SELECT * FROM campaigns LIMIT 5"
pd.read_sql(query, conn)

,Data Source name,Date,Campaign Name,Campaign Effective Status,Ad Set Name,Ad Name,Country Funnel,Geo Location Segment,FB Spent Funnel (INR),spend,...,Website Contacts,Messaging Conversations Started,Adds to Cart Conversion Value (INR),Checkouts Initiated Conversion Value (INR),Adds of Payment Info Conversion Value (INR),Row Count,CTR,CPC,CPM,ROI
0,brand a,2026-01-02 00:00:00,unknown,paused,tof | lal 3-5% | sales | womens,custom | video 2 | eoss | 15th dec 25 – copy,india,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None
1,brand a,2026-01-03 00:00:00,growify | 11th june | tof | abo | india | lal ...,paused,tof | lal 3-5% | sales | womens,custom | video 2 | eoss | 15th dec 25 – copy,india,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None
2,unknown,2026-01-20 00:00:00,growify | 11th june | tof | abo | india | lal ...,paused,unknown,unknown,india,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None
3,brand a,2026-01-01 00:00:00,growify | 11th june | tof | abo | india | lal ...,unknown,tof | lal 3-5% | sales | womens,custom | video 2 | eoss | 15th dec 25 – copy,india,india,0.0,0.0,...,0.0,0.0,5964500.0,0.0,0.0,1.0,None,None,None,None
4,brand a,2026-01-11 00:00:00,growify | 11th june | tof | abo | india | lal ...,paused,unknown,custom | video 2 | eoss | 15th dec 25 – copy,nan,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None


## Create date dimension table

In [33]:
date_range = pd.date_range(start='2025-01-01', end='2026-12-31')

date_df = pd.DataFrame({
    'date': date_range,
    'day': date_range.day,
    'week': date_range.isocalendar().week,
    'month': date_range.month,
    'month_name': date_range.strftime('%B'),
    'quarter': date_range.quarter,
    'year': date_range.year,
    'day_name': date_range.strftime('%A')
})

date_df.to_sql("dim_date", conn, if_exists="replace", index=False)

730

## Verify tables

In [34]:
pd.read_sql("SELECT * FROM dim_date LIMIT 5", conn)

,date,day,week,month,month_name,quarter,year,day_name
0,2025-01-01 00:00:00,1,1,1,January,1,2025,Wednesday
1,2025-01-02 00:00:00,2,1,1,January,1,2025,Thursday
2,2025-01-03 00:00:00,3,1,1,January,1,2025,Friday
3,2025-01-04 00:00:00,4,1,1,January,1,2025,Saturday
4,2025-01-05 00:00:00,5,1,1,January,1,2025,Sunday


# Step 3: Star Schema Design

Designing a simple star schema where:

- **fact_sales** → contains transactional sales data (Shopify)
- **dim_campaign** → contains campaign-level metadata
- **dim_date** → contains date attributes for time-based analysis

Since there is no direct key between campaigns and sales, we use **Date** as a common join key.

This approach allows downstream tools (Power BI, AI tool) to query structured data efficiently.

## Step 3.1: Standardise Column Names

Converting column names into SQL-friendly format:
- lowercase
- remove spaces
- remove brackets

This ensures consistency and avoids SQL errors.

In [35]:
# clean column names

df_shopify.columns = (
    df_shopify.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
)

df_campaign.columns = (
    df_campaign.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [36]:
# check columns name after converting to lowercase
df_campaign.columns
df_shopify.columns

Index(['data_source_name', 'date', 'currency', 'sales_channel',
       'transaction_timestamp', 'order_created_at', 'order_updated_at',
       'order_id', 'order_name', 'country_funnel', 'geo_location_segment',
       'billing_country', 'billing_province', 'billing_city', 'order_tags',
       'product_id', 'product_title', 'product_tags', 'product_type',
       'variant_title', 'gross_sales_inr', 'net_sales_inr', 'total_sales_inr',
       'orders', 'returns_inr', 'return_rate', 'items_sold', 'items_returned',
       'average_order_value_inr', 'new_customer_orders',
       'returning_customer_orders', 'average_items_per_order', 'discounts_inr',
       'row_count', 'sku', 'customer_sale_type', 'customer_id',
       'shipping_country', 'invalid_order_dates', 'invalid_transaction_time',
       'roi_calculated', 'roi_flag'],
      dtype='object')

In [37]:
# insert data into shopify and campaigns tables from shopify and campaign data
df_shopify.to_sql("shopify", conn, if_exists="replace", index=False)
df_campaign.to_sql("campaigns", conn, if_exists="replace", index=False)

10028

In [38]:
# check inserted data into table
pd.read_sql("PRAGMA table_info(shopify);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,data_source_name,TEXT,0,None,0
1,1,date,TEXT,0,None,0
2,2,currency,TEXT,0,None,0
3,3,sales_channel,TEXT,0,None,0
4,4,transaction_timestamp,TEXT,0,None,0
5,5,order_created_at,TEXT,0,None,0
6,6,order_updated_at,TEXT,0,None,0
7,7,order_id,REAL,0,None,0
8,8,order_name,TEXT,0,None,0
9,9,country_funnel,TEXT,0,None,0


In [43]:
# check inserted data into table
pd.read_sql("PRAGMA table_info(campaigns);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,data_source_name,TEXT,0,None,0
1,1,date,TEXT,0,None,0
2,2,campaign_name,TEXT,0,None,0
3,3,campaign_effective_status,TEXT,0,None,0
4,4,ad_set_name,TEXT,0,None,0
5,5,ad_name,TEXT,0,None,0
6,6,country_funnel,TEXT,0,None,0
7,7,geo_location_segment,TEXT,0,None,0
8,8,fb_spent_funnel_(inr),REAL,0,None,0
9,9,spend,REAL,0,None,0


## Step 3.2: Create Fact Table (Sales)

The fact table contains measurable metrics such as:
- revenue
- orders
- items sold

This table will be used for aggregations in Power BI.

In [58]:
query = """
DROP TABLE IF EXISTS fact_sales;

CREATE TABLE fact_sales AS
SELECT
    s.date,
    s.data_source_name,
    c.campaign_name,
    s.billing_country,
    s.sales_channel,

    -- SALES METRICS
    s.total_sales_inr AS revenue,
    s.orders,
    s.items_sold,

    -- MARKETING METRICS
    c.spend AS spend,
    c."clicks_(all)" AS clicks,
    c.impressions AS impressions

FROM shopify s
LEFT JOIN campaigns c
ON s.date = c.date
AND s.data_source_name = c.data_source_name;
"""

conn.executescript(query)

## Step 3.3: Create Campaign Dimension Table

The campaign dimension stores:
- campaign name

This helps in filtering and segmentation.

In [49]:
query = """
DROP TABLE IF EXISTS dim_campaign;

CREATE TABLE dim_campaign AS
SELECT DISTINCT
    campaign_name
FROM campaigns;
"""

conn.executescript(query)

## Step 3.4: Verify Tables

We check if the tables are created correctly and contain data.

In [60]:
pd.read_sql("SELECT * FROM fact_sales LIMIT 5", conn)

,date,data_source_name,campaign_name,billing_country,sales_channel,revenue,orders,items_sold,spend,clicks,impressions
0,2026-01-08 00:00:00,brand a,growify | 2nd december | abo | sale,unknown,online store,0.0,0.0,0.0,193.41,23.0,465.0
1,2026-01-08 00:00:00,brand a,growify | mof + bof | conversion | india,unknown,online store,0.0,0.0,0.0,0.00,0.0,0.0
2,2026-01-08 00:00:00,brand a,growify | mof + bof | conversion | india,unknown,online store,0.0,0.0,0.0,0.00,0.0,0.0
3,2026-01-08 00:00:00,brand a,growify | tof | conversion | india,unknown,online store,0.0,0.0,0.0,126.79,11.0,168.0
4,2026-01-08 00:00:00,brand a,growify | tof | conversion | india,unknown,online store,0.0,0.0,0.0,238.39,21.0,34510.0


In [51]:
pd.read_sql("SELECT * FROM dim_campaign LIMIT 5", conn)

,campaign_name
0,unknown
1,growify | 11th june | tof | abo | india | lal ...
2,growify | 16th june | tof | india | asc | cac ...
3,growify | 23rd sept | tof | abo | sales
4,growify | 2nd december | abo | sale


## Step 4: Power BI Aggregation Query

We aggregate sales performance by platform, channel, region, and month using fact_sales

In [47]:
#  POWER BI AGGREGATION QUERY

query = """
SELECT
    f.data_source_name AS platform,
    f.sales_channel AS channel,
    f.billing_country AS region,
    strftime('%Y-%m', date(f.date)) AS month,

    SUM(f.revenue) AS total_revenue,
    SUM(f.orders) AS total_orders,
    SUM(f.items_sold) AS total_items

FROM fact_sales f

WHERE 
    f.data_source_name IS NOT NULL
    AND f.sales_channel IS NOT NULL
    AND f.date IS NOT NULL
    AND f.billing_country IS NOT NULL

    AND f.data_source_name != 'unknown'
    AND f.sales_channel != 'unknown'
    AND f.billing_country != 'unknown'

GROUP BY
    platform,
    channel,
    region,
    month

ORDER BY
    month, platform, channel
"""

pd.read_sql(query, conn)

,platform,channel,region,month,total_revenue,total_orders,total_items
0,brand a,draft orders,india,2026-01,9.191658e+07,20770.0,3867.0
1,brand a,draft orders,united states,2026-01,1.493800e+05,11.0,11.0
2,brand a,magic checkout - brand a,india,2026-01,1.749515e+09,167725.0,84425.0
3,brand a,online store,canada,2026-01,0.000000e+00,11.0,11.0
4,brand a,online store,india,2026-01,1.509843e+07,55.0,64.0
...,...,...,...,...,...,...,...
85,brand b,online store,saudi arabia,2026-03,1.697160e+07,163.0,244.0
86,brand b,online store,thailand,2026-03,-3.306000e+07,0.0,0.0
87,brand b,online store,united arab emirates,2026-03,6.555000e+06,8208.0,0.0
88,brand b,online store,united kingdom,2026-03,2.741508e+08,9936.0,69.0


## Step 5: AI Query (Dynamic & Flexible Filtering)

This query is designed for an AI tool to dynamically filter data based on user inputs such as date range, platform, region, and channel.

Instead of writing multiple queries, we use optional filters with parameterized conditions. This allows the same query to serve multiple use cases.

Key Features:
- Supports date range filtering
- Supports optional filtering (platform, region, channel)
- Returns aggregated metrics
- Clean and scalable design for AI/analytics use

In [54]:
# STEP 5: AI QUERY
query = """
SELECT
    f.data_source_name AS platform,
    f.sales_channel AS channel,
    f.billing_country AS region,
    strftime('%Y-%m', date(f.date)) AS month,

    SUM(f.revenue) AS total_revenue,
    SUM(f.orders) AS total_orders,
    SUM(f.items_sold) AS total_items

FROM fact_sales f

WHERE
    -- Mandatory date filter
    date(f.date) BETWEEN :start_date AND :end_date

    -- Optional filters
    AND (:platform IS NULL OR f.data_source_name = :platform)
    AND (:region IS NULL OR f.billing_country = :region)
    AND (:channel IS NULL OR f.sales_channel = :channel)

    -- Data quality filters
    AND f.data_source_name IS NOT NULL
    AND f.sales_channel IS NOT NULL
    AND f.billing_country IS NOT NULL
    AND f.data_source_name != 'unknown'
    AND f.sales_channel != 'unknown'
    AND f.billing_country != 'unknown'

GROUP BY
    platform,
    channel,
    region,
    month

ORDER BY
    month
"""

## Example Usage

This query can be executed with different parameter values depending on the requirement.

If a parameter is set to NULL, that filter is ignored.

In [55]:
params = {
    "start_date": "2026-01-01",
    "end_date": "2026-03-31",
    "platform": None,
    "region": None,
    "channel": None
}

pd.read_sql(query, conn, params=params)

,platform,channel,region,month,total_revenue,total_orders,total_items
0,brand a,draft orders,india,2026-01,9.191658e+07,20770.0,3867.0
1,brand a,draft orders,united states,2026-01,1.493800e+05,11.0,11.0
2,brand a,magic checkout - brand a,india,2026-01,1.749515e+09,167725.0,84425.0
3,brand a,online store,canada,2026-01,0.000000e+00,11.0,11.0
4,brand a,online store,india,2026-01,1.509843e+07,55.0,64.0
...,...,...,...,...,...,...,...
85,brand b,online store,saudi arabia,2026-03,1.697160e+07,163.0,244.0
86,brand b,online store,thailand,2026-03,-3.306000e+07,0.0,0.0
87,brand b,online store,united arab emirates,2026-03,6.555000e+06,8208.0,0.0
88,brand b,online store,united kingdom,2026-03,2.741508e+08,9936.0,69.0


## Step 6: Adding Indexes for Query Optimization

Indexes are used to improve query performance by allowing the database to quickly locate rows instead of scanning the entire table.

Since the queries frequently filter on specific columns, adding indexes on those columns will significantly improve performance, especially for large datasets.

### Columns Selected for Indexing

Based on the query patterns, I identified the following frequently used columns:

- date → used in date range filtering
- data_source_name (platform) → used for filtering
- billing_country (region) → used for filtering
- sales_channel (channel) → used for filtering

These columns are part of the WHERE clause in the Power BI and AI queries.

In [57]:
index_query = """
CREATE INDEX IF NOT EXISTS idx_fact_sales_date 
ON fact_sales(date);

CREATE INDEX IF NOT EXISTS idx_fact_sales_platform 
ON fact_sales(data_source_name);

CREATE INDEX IF NOT EXISTS idx_fact_sales_region 
ON fact_sales(billing_country);

CREATE INDEX IF NOT EXISTS idx_fact_sales_channel 
ON fact_sales(sales_channel);
"""
conn.executescript(index_query)

## Why These Indexes?

1. Date Index:
   Improves performance of date range queries (BETWEEN start_date AND end_date)

2. Platform Index:
   Speeds up filtering by platform (e.g., brand a, brand b)

3. Region Index:
   Optimizes queries filtering by country/region

4. Channel Index:
   Helps in segmenting performance by sales channel

Without indexes, the database would perform a full table scan for each query, which is inefficient for large datasets.

With indexes, the database can directly locate relevant rows, reducing query execution time significantly.

## Additional Notes

- Indexes improve read/query performance but may slightly slow down write operations.
- Since this project focuses on analytics (read-heavy workload), indexes are beneficial.
- Chosen indexes align with real-world BI and AI query patterns.